In [1]:
import sys
sys.path.insert(0, '..')
from dependencies import *

envelopes_log = eelbrain.load.unpickle(PROCESSED_PREDICTOR_DIR / f'~processed_envelopes-log.pickle')
subject_model_predictors = eelbrain.load.unpickle(PREDICTOR_DIR / f'~concatenated_predictors.pickle')
durations = get_durations(envelopes_log)
models = get_models(exclude=['envelope_log_8band', 'envelope_log_onset'])

In [2]:
import eelbrain
import numpy as np
from scipy.stats import ttest_rel

# -----------------------------
# Storage
# -----------------------------
r_values = {model: [] for model in models}
r2_values = {model: [] for model in models}

# -----------------------------
# Loop over subjects
# -----------------------------
for subject in SUBJECTS:
    print(f'\n===== Subject {subject} =====')

    eeg = eelbrain.load.unpickle(STIMULUS_DIR / f'{subject}concatenated_eeg.pickle')
    eeg_data = eeg.x  # (channels, time)

    for model in models:
        print(f'\nModel: {model}')

        # Load subject TRF
        trf_subject = eelbrain.load.unpickle(TRF_DIR / subject / f'{subject} {model}.pickle')
        predictors = subject_model_predictors[subject][model]

        # Build stimulus (subject predictors)
        stimulus = predictors

        # Predict EEG
        predicted_subject = eelbrain.convolve(trf_subject.h_scaled, stimulus)
        pred = predicted_subject.x

        # Per-channel correlation
        r_channels = []
        for ch in range(eeg_data.shape[0]):
            # Skip channels with zero variance
            if np.std(eeg_data[ch]) > 0 and np.std(pred[ch]) > 0:
                r_channels.append(np.corrcoef(eeg_data[ch], pred[ch])[0,1])

        # Skip TRF if all channels are invalid
        if len(r_channels) == 0:
            print("All channels invalid for this TRF, skipping.")
            continue

        # Mean across valid channels
        r_mean = np.mean(r_channels)
        r2_mean = r_mean**2

        r_values[model].append(r_mean)
        r2_values[model].append(r2_mean)

        print(f'Subject TRF: r = {r_mean:.4f}, r² = {r2_mean:.4f}')

# -----------------------------
# Summary statistics: Subject TRFs
# -----------------------------
print('\n===== Subject TRF summary =====')
for model in models:
    if len(r_values[model]) == 0:
        print(f'{model}: No valid TRFs!')
        continue
    mean_r = np.mean(r_values[model])
    std_r = np.std(r_values[model])
    mean_r2 = np.mean(r2_values[model])
    std_r2 = np.std(r2_values[model])
    print(f'{model}: r = {mean_r:.4f} ± {std_r:.4f}, r² = {mean_r2:.4f} ± {std_r2:.4f}')



===== Subject S05 =====

Model: envelope_log
Subject TRF: r = 0.0367, r² = 0.0013

Model: envelope_onset
Subject TRF: r = 0.0369, r² = 0.0014

===== Subject S34 =====

Model: envelope_log
Subject TRF: r = 0.0616, r² = 0.0038

Model: envelope_onset
Subject TRF: r = 0.0435, r² = 0.0019

===== Subject S35 =====

Model: envelope_log
Subject TRF: r = 0.0702, r² = 0.0049

Model: envelope_onset
Subject TRF: r = 0.0427, r² = 0.0018

===== Subject S03 =====

Model: envelope_log
Subject TRF: r = 0.0620, r² = 0.0038

Model: envelope_onset
Subject TRF: r = 0.0233, r² = 0.0005

===== Subject S04 =====

Model: envelope_log
Subject TRF: r = 0.0512, r² = 0.0026

Model: envelope_onset
Subject TRF: r = 0.0368, r² = 0.0014

===== Subject S44 =====

Model: envelope_log
Subject TRF: r = 0.0098, r² = 0.0001

Model: envelope_onset
Subject TRF: r = 0.0082, r² = 0.0001

===== Subject S26 =====

Model: envelope_log
Subject TRF: r = -0.0014, r² = 0.0000

Model: envelope_onset
Subject TRF: r = -0.0004, r² = 0.00

In [3]:
t_stat, p_val = ttest_rel(r_values['envelope_log'], r_values['envelope_onset'])
print(f'Envelope log vs. onset: t={t_stat:.2f}, p={p_val:.4f}')

Envelope log vs. onset: t=8.52, p=0.0000


In [5]:
# ------------------------------------------------
# Storage
# ------------------------------------------------
r_values_universal = {model: [] for model in models}
r2_values_universal = {model: [] for model in models}

# ------------------------------------------------
# Loop over models
# ------------------------------------------------
for model in models:

    print(f'\n===== Universal encoder: {model} =====')

    # ----------------------------------------
    # Load universal decoder TRF
    # ----------------------------------------
    trf_universal = eelbrain.load.unpickle(
        TRF_DIR / f'universal-trf-{model}.pickle'
    )

    # ----------------------------------------
    # Loop over subjects
    # ----------------------------------------
    for subject in SUBJECTS:

        print(f'\nSubject: {subject}')

        eeg = eelbrain.load.unpickle(
            STIMULUS_DIR / f'{subject}concatenated_eeg.pickle'
        )

        predictors = subject_model_predictors[subject][model]

        # ----------------------------------------
        # Predict stimulus from EEG
        # ----------------------------------------
        predicted = eelbrain.convolve(trf_universal, eeg)

        # ----------------------------------------
        # Build actual stimulus
        # ----------------------------------------
        #stimulus = sum(predictors) if len(predictors) > 1 else predictors[0]
        stimulus = predictors[0]

        # ----------------------------------------
        # Convert to numpy
        # ----------------------------------------
        env = stimulus.x
        pred = predicted.x

        # ----------------------------------------
        # Correlation
        # ----------------------------------------
        r = np.corrcoef(env, pred)[0,1]
        r2 = r**2

        r_values_universal[model].append(r)
        r2_values_universal[model].append(r2)

        print(f'r = {r:.4f}, r² = {r2:.4f}')

# ------------------------------------------------
# Summary statistics
# ------------------------------------------------
print('\n===== Universal encoder summary =====')

for model in models:
    mean_r = np.mean(r_values_universal[model])
    std_r = np.std(r_values_universal[model])
    mean_r2 = np.mean(r2_values_universal[model])
    std_r2 = np.std(r2_values_universal[model])

    print(f'{model}: r = {mean_r:.4f} ± {std_r:.4f}, r² = {mean_r2:.4f} ± {std_r2:.4f}')


===== Universal encoder: envelope_log =====

Subject: S05
r = -0.0249, r² = 0.0006

Subject: S34
r = 0.0100, r² = 0.0001

Subject: S35
r = -0.0021, r² = 0.0000

Subject: S03
r = 0.0108, r² = 0.0001

Subject: S04
r = 0.0369, r² = 0.0014

Subject: S44
r = -0.0014, r² = 0.0000

Subject: S26
r = -0.0064, r² = 0.0000

Subject: S19
r = 0.0121, r² = 0.0001

Subject: S21
r = 0.0450, r² = 0.0020

Subject: S17
r = 0.0703, r² = 0.0049

Subject: S10
r = 0.0261, r² = 0.0007

Subject: S42
r = 0.0198, r² = 0.0004

Subject: S45
r = -0.0220, r² = 0.0005

Subject: S11
r = 0.0440, r² = 0.0019

Subject: S16
r = 0.1085, r² = 0.0118

Subject: S20
r = 0.0648, r² = 0.0042

Subject: S18
r = 0.0322, r² = 0.0010

Subject: S01
r = 0.0757, r² = 0.0057

Subject: S39
r = 0.0667, r² = 0.0044

Subject: S06
r = 0.0574, r² = 0.0033

Subject: S08
r = 0.0586, r² = 0.0034

Subject: S37
r = 0.0323, r² = 0.0010

Subject: S36
r = -0.0069, r² = 0.0000

Subject: S38
r = -0.0095, r² = 0.0001

Subject: S22
r = 0.0043, r² = 0.000